# 01 — Data Preparation

Load raw TMDB API data, clean fields, write `data/movies_clean.csv`.

In [1]:
import json, pandas as pd
from pathlib import Path

RAW = Path('../data/movies_raw.json')
OUT = Path('../data/movies_clean.csv')

with open(RAW) as f:
    raw = json.load(f)

print(f'Loaded {len(raw)} movies')
df = pd.DataFrame(raw)
df.head(2)

Loaded 2000 movies


,id,title,overview,tagline,genres,keywords,vote_average,vote_count,release_date,runtime
0,157336,Interstellar,The adventures of a group of explorers who mak...,Mankind was born on Earth. It was never meant ...,"[Adventure, Drama, Science Fiction]","[spacecraft, race against time, artificial int...",8.483,40400,2014-11-05,169
1,27205,Inception,"Cobb, a skilled thief who commits corporate es...",Your mind is the scene of the crime.,"[Action, Science Fiction, Adventure]","[mission, dreams, kidnapping, spy, allegory, i...",8.372,39620,2010-07-15,148


In [2]:
# --- Parse and flatten ---
df['genres'] = df['genres'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
df['keywords'] = df['keywords'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year.fillna(0).astype(int)

# Vote bucket for keyword filter in minsearch
def vote_bucket(v):
    if v >= 7.5: return 'high'
    if v >= 6.0: return 'medium'
    return 'low'
df['vote_bucket'] = df['vote_average'].apply(vote_bucket)

# Keep only rows with a meaningful overview
df = df[df['overview'].str.strip().str.len() > 20]

# Final column selection
cols = ['id','title','overview','tagline','genres','keywords',
        'vote_average','vote_count','release_year','runtime','vote_bucket']
df = df[cols].reset_index(drop=True)
print(df.shape)
df.head(3)

(2000, 11)


,id,title,overview,tagline,genres,keywords,vote_average,vote_count,release_year,runtime,vote_bucket
0,157336,Interstellar,The adventures of a group of explorers who mak...,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","spacecraft, race against time, artificial inte...",8.483,40400,2014,169,high
1,27205,Inception,"Cobb, a skilled thief who commits corporate es...",Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","mission, dreams, kidnapping, spy, allegory, in...",8.372,39620,2010,148,high
2,24428,The Avengers,When an unexpected enemy emerges and threatens...,Some assembly required.,"Science Fiction, Action, Adventure","new york city, superhero, shield, based on com...",8.058,38839,2012,143,high


In [3]:
# Sanity checks
print('Null counts:')
print(df.isnull().sum())
print('\nVote bucket distribution:')
print(df['vote_bucket'].value_counts())
print('\nGenres sample:', df['genres'].iloc[0])

Null counts:
id              0
title           0
overview        0
tagline         0
genres          0
keywords        0
vote_average    0
vote_count      0
release_year    0
runtime         0
vote_bucket     0
dtype: int64

Vote bucket distribution:
vote_bucket
medium    1352
high       503
low        145
Name: count, dtype: int64

Genres sample: Adventure, Drama, Science Fiction


In [4]:
df.to_csv(OUT, index=False)
print(f'Saved {len(df)} rows to {OUT}')

Saved 2000 rows to ../data/movies_clean.csv
